# Sprint 1 — 會員購物週期計算與沉睡客識別

- 資料來源：`91APP_Dataset(main)/Order_TG.csv`
- 條件：`StatusDef = 'Finish'` 且 `OrderDateTime < '2023-09-01'`
- 輸出：`output/sprint1/member_cycle.parquet`、`output/sprint1/dormant_members.parquet`

> ⚠️ **Cell 1、2、4、5 會產生 / 覆寫 Parquet，已輸出則不需重跑。Cell 3、6 僅驗證用。**

In [26]:
import duckdb
import os

con = duckdb.connect()

ORDER_PATH  = '91APP_Dataset(main)/Order_TG.csv'
OUTPUT_PATH = 'output/sprint1/member_cycle.parquet'

os.makedirs('output/sprint1', exist_ok=True)

## Cell 1 — 篩選條件：Finish & < 2023-09-01 ｜ 會員統計與 is_imputed 標記

篩選 `StatusDef='Finish'` 且 `OrderDateTime < 2023-09-01` 的訂單，建立 `orders_filtered` VIEW。
統計總筆數與涵蓋會員數，並為每位會員標記 `is_imputed`（購買僅 1 次 → True，≥ 2 次 → False）。

> 📄 **輸出**：不直接輸出檔案（建立 VIEW 供後續 Cell 使用）

In [27]:
# 篩選：StatusDef = 'Finish' 且 OrderDateTime < 2023-09-01
con.execute(f"""
CREATE OR REPLACE VIEW orders_filtered AS
SELECT ShopMemberId,
       CAST(OrderDateTime AS DATE) AS order_date
FROM   read_csv_auto('{ORDER_PATH}')
WHERE  StatusDef     = 'Finish'
  AND  OrderDateTime < '2023-09-01'
""")

total_rows    = con.execute("SELECT COUNT(*) FROM orders_filtered").fetchone()[0]
member_count  = con.execute("SELECT COUNT(DISTINCT ShopMemberId) FROM orders_filtered").fetchone()[0]

print(f"篩選後總筆數   : {total_rows:,}")
print(f"涵蓋不重複會員 : {member_count:,}")

# 每位會員：購買次數、最後購買日、is_imputed（購買 1 次 = True）
con.execute("""
CREATE OR REPLACE VIEW member_stats AS
SELECT
    ShopMemberId,
    COUNT(DISTINCT order_date)  AS purchase_count,
    MAX(order_date)             AS last_purchase_date,
    (COUNT(DISTINCT order_date) = 1) AS is_imputed
FROM orders_filtered
GROUP BY ShopMemberId
""")

single, multi = con.execute("""
SELECT
    SUM(CASE WHEN is_imputed = true  THEN 1 END),
    SUM(CASE WHEN is_imputed = false THEN 1 END)
FROM member_stats
""").fetchone()

print(f"\n購買 1 次（is_imputed=True）  : {int(single):>10,}")
print(f"購買 2 次以上（is_imputed=False）: {int(multi):>10,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

篩選後總筆數   : 8,086,587
涵蓋不重複會員 : 2,217,958


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


購買 1 次（is_imputed=True）  :  1,087,805
購買 2 次以上（is_imputed=False）:  1,130,153


## Cell 2 — 購物週期計算（≥ 2 次取相鄰間隔中位數，1 次填 NULL）與輸出

- 購買 ≥ 2 次的會員：以 LAG 計算相鄰購買日間隔，取各人的中位數作為 `personal_cycle_days`
- 購買 1 次（`is_imputed=True`）：`personal_cycle_days` 維持 NULL（Phase 2 的 Cell 4 才填入）
- 合併 `member_stats` 與週期計算結果，輸出 Parquet

> 📄 **輸出**：`output/sprint1/member_cycle.parquet`（2,217,958 筆）

In [28]:
# 購買 >= 2 次：計算相鄰購買日的間隔中位數
# 購買 = 1 次：personal_cycle_days 維持 NULL
con.execute("""
CREATE OR REPLACE VIEW member_personal_cycle AS
WITH daily_orders AS (
    SELECT DISTINCT ShopMemberId, order_date
    FROM   orders_filtered
),
with_lag AS (
    SELECT
        ShopMemberId,
        order_date,
        LAG(order_date) OVER (PARTITION BY ShopMemberId ORDER BY order_date) AS prev_date
    FROM daily_orders
),
gaps AS (
    SELECT
        ShopMemberId,
        DATEDIFF('day', prev_date, order_date) AS gap_days
    FROM with_lag
    WHERE prev_date IS NOT NULL
)
SELECT
    ShopMemberId,
    MEDIAN(gap_days) AS personal_cycle_days
FROM gaps
GROUP BY ShopMemberId
""")

# 全體中位數（只用真實有週期的人算，備用）
global_median = con.execute("""
SELECT MEDIAN(personal_cycle_days) FROM member_personal_cycle
""").fetchone()[0]
print(f"全體購物週期中位數（>= 2 次）：{global_median} 天")

# 輸出 Parquet：購買 1 次的 personal_cycle_days 保持 NULL
con.execute(f"""
COPY (
    SELECT
        s.ShopMemberId,
        s.purchase_count,
        s.last_purchase_date,
        s.is_imputed,
        c.personal_cycle_days          -- is_imputed=True 者為 NULL
    FROM member_stats               AS s
    LEFT JOIN member_personal_cycle AS c
        ON s.ShopMemberId = c.ShopMemberId
    ORDER BY s.ShopMemberId
)
TO '{OUTPUT_PATH}' (FORMAT PARQUET)
""")
print(f"輸出完成：{OUTPUT_PATH}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

全體購物週期中位數（>= 2 次）：63.0 天


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

輸出完成：output/sprint1/member_cycle.parquet


## Cell 3 — 驗證輸出（member_cycle.parquet）

讀取已輸出的 `member_cycle.parquet`，確認總人數、兩群分布、週期統計值符合預期，並印出前 5 筆樣本。

> ℹ️ **純驗證用，不產生輸出檔案，不需重跑。**

In [30]:
verify = con.execute(f"""
SELECT
    COUNT(*)                                              AS total_members,
    SUM(CASE WHEN is_imputed = true  THEN 1 END)         AS single_purchase,
    SUM(CASE WHEN is_imputed = false THEN 1 END)         AS repeat_purchase,
    MIN(personal_cycle_days)                             AS cycle_min,
    MAX(personal_cycle_days)                             AS cycle_max,
    MEDIAN(personal_cycle_days)                          AS cycle_median
FROM read_parquet('{OUTPUT_PATH}')
""").df()

print("=== 驗證統計 ===")
print(verify.to_string(index=False))

print("\n--- 前 5 筆（is_imputed=True，週期填補）---")
print(con.execute(f"""
    SELECT * FROM read_parquet('{OUTPUT_PATH}')
    WHERE is_imputed = true LIMIT 5
""").df().to_string(index=False))

print("\n--- 前 5 筆（is_imputed=False，真實週期）---")
print(con.execute(f"""
    SELECT * FROM read_parquet('{OUTPUT_PATH}')
    WHERE is_imputed = false LIMIT 5
""").df().to_string(index=False))

=== 驗證統計 ===
 total_members  single_purchase  repeat_purchase  cycle_min  cycle_max  cycle_median
       2217958        1087805.0        1130153.0        1.0      607.0          63.0

--- 前 5 筆（is_imputed=True，週期填補）---
                                ShopMemberId  purchase_count last_purchase_date  is_imputed  personal_cycle_days
+++5BolP+NGDUoUla9BhIHR1ksOsqgyN9BOYom76uBE=               1         2023-06-17        True                  NaN
+++Dh9KF4vYSWSC5koKJoll/uq+Gx26k0KH7k0wK8tQ=               1         2023-06-08        True                  NaN
+++UwQlHcTWUFf8O9pbESg5NLIVW+mLA2HCiWn5wW7I=               1         2023-08-21        True                  NaN
++/5ZNIDRirrt0vNsNvRehk1CZ80wVVYYoLNMBuazfk=               1         2022-11-08        True                  NaN
++/GEyEBfTFP04WP18rpVll8Sip4lVUGR7AA2Fi86jk=               1         2022-09-22        True                  NaN

--- 前 5 筆（is_imputed=False，真實週期）---
                                ShopMemberId  purchase_count last_

# ── Sprint 1 Phase 2 ── 沉睡客識別

以 IQR 方法去除高頻假沉睡客，重新計算全體購物週期中位數，最終識別出沉睡客名單。

## Cell 4 — IQR 下限篩除高頻假沉睡客，重新計算全體中位數並填入 is_imputed=True

對 `is_imputed=False` 群體計算 Q1/Q3/IQR，以 **Q1（29 天）** 為下限，排除高頻短週期會員。
以篩選後群體重新計算全體中位數（88.5 天），填入 `is_imputed=True` 的 NULL 欄位。
覆寫輸出 `member_cycle.parquet`。

> 📄 **輸出**：`output/sprint1/member_cycle.parquet`（覆寫，1,942,165 筆）

In [31]:
CYCLE_PATH = 'output/sprint1/member_cycle.parquet'

# 1. 計算 is_imputed=False 群的 IQR
q1, q3, iqr = con.execute(f"""
SELECT
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY personal_cycle_days),
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY personal_cycle_days),
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY personal_cycle_days)
    - PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY personal_cycle_days)
FROM read_parquet('{CYCLE_PATH}')
WHERE is_imputed = false
""").fetchone()

lower_bound = q1   # Q1 作為下限，篩除高頻購買的假沉睡客

print(f"=== is_imputed=False 購買週期 IQR ===")
print(f"  Q1  : {q1} 天")
print(f"  Q3  : {q3} 天")
print(f"  IQR : {iqr} 天")
print(f"  下限（Q1）: {lower_bound} 天  ← 篩除 personal_cycle_days < Q1 的高頻會員")

# 2. 以 Q1 為下限，重新計算全體購買中位數
new_global_median = con.execute(f"""
SELECT MEDIAN(personal_cycle_days)
FROM read_parquet('{CYCLE_PATH}')
WHERE is_imputed = false
  AND personal_cycle_days >= {lower_bound}
""").fetchone()[0]

print(f"\n全體購物週期中位數（購買 >= 2 次 & cycle >= Q1）：{new_global_median} 天")

# 3. 覆蓋輸出：
#    - is_imputed=False & cycle >= Q1 → 保留真實週期
#    - is_imputed=True               → 填入新全體中位數
#    - is_imputed=False & cycle < Q1 → 排除
con.execute(f"""
COPY (
    SELECT
        ShopMemberId,
        purchase_count,
        last_purchase_date,
        is_imputed,
        CASE
            WHEN is_imputed = false THEN personal_cycle_days
            WHEN is_imputed = true  THEN {new_global_median}
        END AS personal_cycle_days
    FROM read_parquet('{CYCLE_PATH}')
    WHERE is_imputed = true
       OR (is_imputed = false AND personal_cycle_days >= {lower_bound})
    ORDER BY ShopMemberId
)
TO '{CYCLE_PATH}' (FORMAT PARQUET)
""")

# 4. 確認結果
cnt_total, cnt_true, cnt_false = con.execute(f"""
SELECT COUNT(*),
       SUM(CASE WHEN is_imputed=true  THEN 1 END),
       SUM(CASE WHEN is_imputed=false THEN 1 END)
FROM read_parquet('{CYCLE_PATH}')
""").fetchone()

print(f"\n覆蓋輸出完成：{CYCLE_PATH}")
print(f"  總人數           : {cnt_total:,}")
print(f"  is_imputed=True  : {int(cnt_true):,}  （填入 {new_global_median} 天）")
print(f"  is_imputed=False : {int(cnt_false):,}  （cycle >= {lower_bound} 天）")

=== is_imputed=False 購買週期 IQR ===
  Q1  : 29.0 天
  Q3  : 129.5 天
  IQR : 100.5 天
  下限（Q1）: 29.0 天  ← 篩除 personal_cycle_days < Q1 的高頻會員

全體購物週期中位數（購買 >= 2 次 & cycle >= Q1）：88.5 天

覆蓋輸出完成：output/sprint1/member_cycle.parquet
  總人數           : 1,942,165
  is_imputed=True  : 1,087,805  （填入 88.5 天）
  is_imputed=False : 854,360  （cycle >= 29.0 天）


## Cell 5 — 新增 dormant_threshold_date、days_since_last，篩選沉睡客

計算每位會員的 `dormant_threshold_date = last_purchase_date + personal_cycle_days × 1.5`。
篩選條件：`dormant_threshold_date < 2023-09-01`（超過 1.5 個購物週期未回購）。
新增 `days_since_last`（距基準日的沉睡天數），輸出沉睡客名單。

> 📄 **輸出**：`output/sprint1/dormant_members.parquet`（1,112,087 筆）

In [32]:
DORMANT_PATH = 'output/sprint1/dormant_members.parquet'
REF_DATE     = '2023-09-01'

# 沉睡條件：dormant_threshold_date（last_purchase + cycle × 1.5）< 基準日 2023-09-01
con.execute(f"""
COPY (
    SELECT
        ShopMemberId,
        purchase_count,
        last_purchase_date,
        is_imputed,
        personal_cycle_days,
        (last_purchase_date
            + INTERVAL (CAST(personal_cycle_days * 1.5 AS INTEGER) || ' days')
        )::DATE                                                    AS dormant_threshold_date,
        DATEDIFF('day', last_purchase_date, DATE '{REF_DATE}')     AS days_since_last
    FROM read_parquet('{CYCLE_PATH}')
    WHERE
        (last_purchase_date
            + INTERVAL (CAST(personal_cycle_days * 1.5 AS INTEGER) || ' days')
        )::DATE < DATE '{REF_DATE}'
    ORDER BY ShopMemberId
)
TO '{DORMANT_PATH}' (FORMAT PARQUET)
""")

stats = con.execute(f"""
SELECT
    COUNT(*),
    SUM(CASE WHEN is_imputed = true  THEN 1 END),
    SUM(CASE WHEN is_imputed = false THEN 1 END),
    MIN(days_since_last),
    MEDIAN(days_since_last),
    MAX(days_since_last)
FROM read_parquet('{DORMANT_PATH}')
""").fetchone()

print(f"=== 沉睡客識別結果 ===")
print(f"沉睡客總人數       : {stats[0]:,}")
print(f"  is_imputed=True  : {int(stats[1]):,}  （填補週期）")
print(f"  is_imputed=False : {int(stats[2]):,}  （真實週期）")
print(f"\n沉睡時間（days_since_last）")
print(f"  最小值 : {stats[3]} 天")
print(f"  中位數 : {stats[4]} 天")
print(f"  最大值 : {stats[5]} 天")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== 沉睡客識別結果 ===
沉睡客總人數       : 1,112,087
  is_imputed=True  : 788,844  （填補週期）
  is_imputed=False : 323,243  （真實週期）

沉睡時間（days_since_last）
  最小值 : 45 天
  中位數 : 304.0 天
  最大值 : 608 天


## Cell 6 — 資料驗證

驗證 `member_cycle.parquet` 與 `dormant_members.parquet`：
- 確認 NULL 欄位數為 0
- 確認 `dormant_threshold_date` 全部小於基準日 `2023-09-01`
- 印出前 5 筆預覽

> ℹ️ **純驗證用，不產生輸出檔案，不需重跑。**

In [33]:
print("=== 驗證 member_cycle.parquet ===")
cycle_check = con.execute(f"""
SELECT
    COUNT(*),
    COALESCE(SUM(CASE WHEN personal_cycle_days IS NULL THEN 1 END), 0),
    MIN(personal_cycle_days),
    MAX(personal_cycle_days),
    MEDIAN(personal_cycle_days)
FROM read_parquet('{CYCLE_PATH}')
""").fetchone()
print(f"  總人數               : {cycle_check[0]:,}")
print(f"  NULL cycle 數        : {cycle_check[1]}  ({'OK' if cycle_check[1]==0 else 'WARNING'})")
print(f"  cycle min/max/median : {cycle_check[2]} / {cycle_check[3]} / {cycle_check[4]} 天")

print("\n=== 驗證 dormant_members.parquet ===")
dormant_check = con.execute(f"""
SELECT
    COUNT(*),
    COALESCE(SUM(CASE WHEN personal_cycle_days IS NULL THEN 1 END), 0),
    COALESCE(SUM(CASE WHEN dormant_threshold_date IS NULL THEN 1 END), 0),
    COALESCE(SUM(CASE WHEN dormant_threshold_date >= DATE '{REF_DATE}' THEN 1 END), 0)
FROM read_parquet('{DORMANT_PATH}')
""").fetchone()
print(f"  沉睡客總人數                  : {dormant_check[0]:,}")
print(f"  NULL personal_cycle_days    : {dormant_check[1]}  ({'OK' if dormant_check[1]==0 else 'WARNING'})")
print(f"  NULL dormant_threshold_date : {dormant_check[2]}  ({'OK' if dormant_check[2]==0 else 'WARNING'})")
print(f"  threshold >= 基準日（應為 0） : {dormant_check[3]}  ({'OK' if dormant_check[3]==0 else 'WARNING'})")

print("\n--- 前 5 筆預覽 ---")
print(con.execute(f"SELECT * FROM read_parquet('{DORMANT_PATH}') LIMIT 5").df().to_string(index=False))

=== 驗證 member_cycle.parquet ===
  總人數               : 1,942,165
  NULL cycle 數        : 0  (OK)
  cycle min/max/median : 29.0 / 607.0 / 88.5 天

=== 驗證 dormant_members.parquet ===
  沉睡客總人數                  : 1,112,087
  NULL personal_cycle_days    : 0  (OK)
  NULL dormant_threshold_date : 0  (OK)
  threshold >= 基準日（應為 0） : 0  (OK)

--- 前 5 筆預覽 ---
                                ShopMemberId  purchase_count last_purchase_date  is_imputed  personal_cycle_days dormant_threshold_date  days_since_last
++/5ZNIDRirrt0vNsNvRehk1CZ80wVVYYoLNMBuazfk=               1         2022-11-08        True                 88.5             2023-03-21              297
++/GEyEBfTFP04WP18rpVll8Sip4lVUGR7AA2Fi86jk=               1         2022-09-22        True                 88.5             2023-02-02              344
++0gUnGIpO99ATqFWxe6pFzVLParYCeiSU2yDvu5xro=               1         2023-03-31        True                 88.5             2023-08-11              154
++0yUMHNKsEGJ8uqhNLRk/bbGwzh4vlZHWG69pA

---
## Sprint 1 總結

### 程式碼流程

| Cell | 步驟 | 說明 |
|------|------|------|
| Cell 1 | 資料篩選與會員統計 | 從 Order_TG.csv 篩選 `StatusDef='Finish'` 且 `OrderDateTime < 2023-09-01`，統計總筆數、會員數，並標記 `is_imputed`（購買 1 次 = True） |
| Cell 2 | 購物週期計算 | 購買 ≥ 2 次的會員計算相鄰購買日間隔中位數作為個人週期；購買 1 次維持 NULL |
| Cell 3 | 驗證 member_cycle.parquet | 確認欄位、人數、兩群分布正確 |
| Cell 4 | IQR 篩除高頻假沉睡客 | 對 `is_imputed=False` 計算 Q1/Q3/IQR，以 **Q1 為下限**排除高頻短週期會員；以篩選後剩餘群體重新計算全體中位數，填入 `is_imputed=True` 的 NULL |
| Cell 5 | 沉睡客識別 | 計算 `dormant_threshold_date = last_purchase_date + personal_cycle_days × 1.5`，篩選條件：門檻日 < 基準日 `2023-09-01` |
| Cell 6 | 驗證與輸出 | 確認無 NULL、門檻日全部 < 基準日，輸出 `dormant_members.parquet` |

---

### 沉睡客篩選定義

- **基準日**：`2023-09-01`（觀察期起點）
- **母體**：在基準日前有過至少一筆 `Finish` 訂單的會員
- **個人購物週期**：
  - 購買 ≥ 2 次 → 相鄰購買日間隔的中位數（排除 cycle < Q1 的高頻異常）
  - 購買 1 次 → 填入篩選後群體的全體中位數（imputed）
- **沉睡門檻**：`last_purchase_date + personal_cycle_days × 1.5 < 2023-09-01`
  - 超過 1.5 個購物週期未回購，視為進入沉睡狀態